# Exploratory Data Analysis

Packages

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import math
import os

# Custom packages
from filter import FilterDF as fdf
from benchmarks import ParetoAnalysis as pa
from benchmarks import AccuracyCalculation as ac
from integrity_fixes import DataFixer as fix, DataExporter as exporter

In [ ]:
%load_ext autoreload
%autoreload 2

Load formatted data

In [ ]:
%store -r static_reference_data
%store -r restaurant_sales_data

## Static Reference Data Exploration

### Retrieval

Retrieve the four static reference data frames

In [ ]:
# Promotional items (30 rows)
before_after_details = static_reference_data['before_after_details'].copy()

# Specific customer information: for matching customers with orders
customers = static_reference_data['customers'].copy()

# Menu items for all restaurants: for matching plant-based labels with orders
items_tagged = static_reference_data['items_tagged'].copy()

# Restaurant details (30 rows)
locations = static_reference_data['locations'].copy()

# Check if location_id is a column to prevent attempting to a remove a column that isn't there
if 'location_id' in before_after_details.columns:
    before_after_details.index = before_after_details['location_id']
    before_after_details.drop(columns=['location_id'], inplace=True)

# Check if location_id is a column to prevent attempting to a remove a column that isn't there
if 'location_id' in locations.columns:
    locations.index = locations['location_id']
    locations.drop(columns=['location_id'], inplace=True)

# List restaurant ids
location_ids = locations.index.tolist()

# Relabeled items
items_tagged_new = pd.read_excel("top_items_tagged.xlsx")

items_tagged_new['item_description'] = ''

items_tagged_new = items_tagged_new[['id', 'location_id', 'item_name', 'item_description', 'item_type', 'dish_category', 'ingredients', 'is_plant_based', 'brand']]

batch_2_restaurants = fdf(items_tagged).filter('batch', 2)['location_id'].unique().tolist()

items_tagged_new['batch'] = np.where(items_tagged_new['location_id'].isin(batch_2_restaurants), 2, 1) 

items_given_restaurant_list = []

for loc_id in location_ids:
    
    items_given_restaurant = fdf(items_tagged).filter('location_id', loc_id)
    new_items = fdf(items_tagged_new).filter('location_id', loc_id)
    new_items_list = new_items['item_name'].unique().tolist()
    old_items = fdf(items_given_restaurant).filter('item_name', new_items_list, exclude=True)
    items_given_restaurant_list.append(pd.concat([old_items, new_items]))

items_tagged = pd.concat(items_given_restaurant_list)

In [ ]:
# Sort
restaurant_sales_data['2HRX9P6HKXA8V'] = restaurant_sales_data['2HRX9P6HKXA8V'].reset_index().sort_values('created_at').set_index('created_at')

### Menu Stats

General

In [ ]:
items_tagged.columns.tolist()

In [ ]:
items_tagged['item_type'].value_counts()

In [ ]:
items_tagged['dish_category'].value_counts()

In [ ]:
items_tagged['ingredients'].value_counts()

In [ ]:
items_tagged['brand'].value_counts()

Specific

In [ ]:
# Create a filter dataframe for easier filtering
items_tagged_fdf = fdf(items_tagged)

# Filter out alcohol
items_tagged_no_alcohol = items_tagged_fdf.filter('dish_category', 'Alcohol', exclude=True)
items_tagged_no_alcohol_fdf = fdf(items_tagged_no_alcohol)

plant_based = items_tagged_no_alcohol_fdf.filter('is_plant_based', 'yes')

### Menu Visuals

In [ ]:
plt.bar(items_tagged['is_plant_based'].value_counts().index, items_tagged['is_plant_based'].value_counts())
plt.title("Total Menu Items: Is it Plant-Based?")
plt.show()

In [ ]:
plt.bar(items_tagged['item_type'].value_counts().index, items_tagged['item_type'].value_counts())
plt.title("Total Menu Items: Item Types")
plt.show()

In [ ]:
plt.bar(items_tagged['dish_category'].value_counts()[:20].index, items_tagged['dish_category'].value_counts()[:20])
plt.title("Total Menu Items: Dish Categories")
plt.xticks(rotation = 75)
plt.show()

In [ ]:
plt.bar(items_tagged_no_alcohol['is_plant_based'].value_counts().index, items_tagged_no_alcohol['is_plant_based'].value_counts(), color="red")
plt.title("Menu Items without Alcohol: Is it Plant-Based?")
plt.show()

In [ ]:
plt.bar(items_tagged_no_alcohol['item_type'].value_counts().index, items_tagged_no_alcohol['item_type'].value_counts(), color="red")
plt.title("Menu Items without Alcohol: Item Types")
plt.show()

In [ ]:
plt.bar(items_tagged_no_alcohol['dish_category'].value_counts()[:20].index, items_tagged_no_alcohol['dish_category'].value_counts()[:20], color="red")
plt.title("Menu Items without Alcohol: Dish Categories")
plt.xticks(rotation = 75)
plt.show()

In [ ]:
plt.bar(plant_based['item_type'].value_counts().index, plant_based['item_type'].value_counts(), color="green")
plt.title("Plant-Based Menu Items without Alcohol: Item Types")
plt.show()

In [ ]:
plt.bar(plant_based['dish_category'].value_counts()[:20].index, plant_based['dish_category'].value_counts()[:20], color="green")
plt.title("Plant-Based Menu Items without Alcohol: Dish Categories")
plt.xticks(rotation = 75)
plt.show()

## Restaurant Sales Data Exploration

### Retrieval

Retrieve the location ids

In [ ]:
location_ids = list(restaurant_sales_data.keys())

### General

Columns

In [ ]:
restaurant_sales_data[location_ids[0]].columns.tolist()

Timeframes

In [ ]:
list_of_timeframes = []

# Find the time difference
for location_id, df in tqdm(restaurant_sales_data.items()):

    # Find the time difference
    timedelta = df.index[-1] - df.index[0]

    # Convert to days, then to years
    years = timedelta.days / 365.25

    # Append
    list_of_timeframes.append(years)

# Turn into an np array for finding median, mean, and std
timeframes = np.array(list_of_timeframes)

# Display
print("Median: {:.2f} year range".format(np.median(timeframes)))
print("Mean: {:.2f} year range".format(np.mean(timeframes)))
print("SD: {:.2f} years".format(np.std(timeframes)))
print("Restaurants with less than a 2 year range: {}".format((timeframes < 2).sum()))

## Data Integrity Checks

### Static Menu Data

In [ ]:
# Count duplicates by different primary keys
menu_id_duplicates = items_tagged['id'].duplicated().sum()
menu_name_duplicates_count = items_tagged[['location_id', 'item_name']].duplicated().sum()
menu_name_duplicates_total = items_tagged[['location_id', 'item_name']].duplicated(keep=False).sum()

print(menu_id_duplicates, menu_name_duplicates_count, menu_name_duplicates_total)

### Restaurant Sales Data

In [ ]:
# Prepare dicts for exact duplicate sales data and a summary
exact_duplicates_dict = {}
id_duplicates_dict = {}
exact_duplicates_summary_list = []

# Prepare dicts for duplicate item names with different unit prices data and a summary
distinct_prices_data_dict = {}
items_with_distinct_unit_prices_dict = {}
distinct_prices_summary_list = []

# Capitalization
capitalization_summary_list = []

# Fractional Quantities
fractional_quantities_dict = {}

# For every restaurant
for location_id, df in tqdm(restaurant_sales_data.items()):

    fix_df = fix(df, location_id)

    # Exact duplicate rows
    exact_duplicates, id_duplicates, non_exact_id_duplicates, exact_duplicate_summary_row = fix_df.find_exact_duplicates(primary_key=['unique_id'])
    
    # Add/Append
    exact_duplicates_dict[location_id] = exact_duplicates
    id_duplicates_dict[location_id] = id_duplicates
    exact_duplicates_summary_list.append(exact_duplicate_summary_row)


    # Items with distinct unit prices
    distinct_unit_prices_data, item_name_list, pricing_discrepancy_summary_row = fix_df.find_pricing_discrepancies()
    
    # Add/Append
    distinct_prices_data_dict[location_id] = distinct_unit_prices_data
    items_with_distinct_unit_prices_dict[location_id] = item_name_list
    distinct_prices_summary_list.append(pricing_discrepancy_summary_row)


    # Items with distinct capitalizations
    capitalization_summary = fix_df.find_capitalization_duplicates()

    # Append
    capitalization_summary_list.append(capitalization_summary)


    # Restaurants with fractional quantities
    fractional_quantities = fix_df.find_fractional_quantities()

    # Add
    fractional_quantities_dict[location_id] = fractional_quantities

## Data Integrity Logs

### Static Menu Data

In [ ]:
menu_duplicates = items_tagged[items_tagged[['location_id', 'item_name']].duplicated(keep=False)].sort_values(['location_id','item_name'])
exporter.export_csv(menu_duplicates, 'example_palate_data_issues/menu_duplicates.csv')

### Restaurant Sales Data

In [ ]:
# Create summary dataframes
exact_duplicates_summary = pd.DataFrame(exact_duplicates_summary_list)
distinct_prices_summary = pd.DataFrame(distinct_prices_summary_list)
capitalization_summary = pd.DataFrame(capitalization_summary_list)

# Print
print((exact_duplicates_summary['rows_delivered'].sum(), exact_duplicates_summary['unique_rows_received'].sum()))

# Export
exporter.export_csv(exact_duplicates_summary, 'example_palate_data_issues/exact_duplicates_summary.csv')
# exporter.export_list(exact_duplicates_list, 'example_palate_data_issues/exact_duplicates.xlsx') <------------ TOO LARGE
exporter.export_csv(distinct_prices_summary, 'example_palate_data_issues/distinct_prices_summary.csv')
# exporter.export_list(distinct_prices_data_list, 'example_palate_data_issues/distinct_prices.xlsx')
exporter.export_csv(fractional_quantities_dict['75WYSXR9QBK5M'], 'example_palate_data_issues/fractional_quantities.csv')
exporter.export_csv(capitalization_summary, 'example_palate_data_issues/capitalization_discrepancies.csv')

## Data Integrity Fixes

### Static Menu Data

Duplicates and NaN

In [ ]:
# Initialize fixer
fix_items_tagged = fix(items_tagged)

# Fix it
fix_items_tagged.remove_nans()
fix_items_tagged.remove_capitalization_duplicates(['item_name','location_id'])

# Retrieve fixed df
items_tagged = fix_items_tagged.get_df()

# Make location uppercase
items_tagged.loc[:,'location_id'] = items_tagged.loc[:,'location_id'].str.upper()

Remove Useless Columns

In [ ]:
# If not removed yet
if 'id' in items_tagged.columns:
    # Restaurant item ids
    restaurant_item_ids = set()
    for location_id, df in restaurant_sales_data.items():
        restaurant_item_ids.union(df['unique_id'].tolist())
        
    # Menu ids
    print(set(items_tagged['id'].tolist()).intersection(restaurant_item_ids))

    # Useless, so remove
    items_tagged.drop(axis=1, labels=['id'], inplace=True)

    for location_id, df in tqdm(restaurant_sales_data.items()):
        df.drop(axis=1, labels=['unique_id'], inplace=True)
        restaurant_sales_data[location_id] = df

### Restaurant Sales Data

Duplicates and NaN

In [ ]:
# For every restaurant
for location_id, df in tqdm(restaurant_sales_data.items()):

    # Fix it
    df = df[~(df['item_name'] == 'nan')]
    df.drop_duplicates(inplace=True)
    df.dropna(inplace=True)
    df.loc[:,'item_name'] = df.loc[:,'item_name'].str.title()

    restaurant_sales_data[location_id] = df


Remove Non Food Items

In [ ]:
%store -r sales_to_remove
restaurant_sales_data['3AXDVZJYN9DRS'] = fdf(restaurant_sales_data['3AXDVZJYN9DRS']).filter('item_name', sales_to_remove[0], exclude=True)
restaurant_sales_data['ED5J990H5VAZT'] = fdf(restaurant_sales_data['ED5J990H5VAZT']).filter('item_name', sales_to_remove[1], exclude=True)

Fixing Encoding Errors

In [ ]:
%store -r encodings_df
for row in encodings_df.iterrows():
    items_tagged['item_name'].replace(to_replace=row[1][0], value=row[1][1], inplace=True)

### Customer Data

In [ ]:
restaurant_sales_data[location_ids[0]].head()

In [ ]:
customers.drop_duplicates(['location_id', 'customer_id'], inplace=True)
customers.reset_index(inplace=True)a

## Merging Data

Merging the sales data and the menu data

In [ ]:
# Check if 'items_tagged' has unique pairs of 'item_name' and 'location_id'
if items_tagged.duplicated(subset=['item_name', 'location_id']).any():
    raise ValueError("Duplicates found in 'items_tagged' for the combination of 'item_name' and 'location_id'")

merged_sales_and_menu = {}
for location_id, df in tqdm(restaurant_sales_data.items()):

    # Copy to prevent overwriting
    df_copy = df.copy()
    items_tagged_copy = items_tagged.copy()

    # For ease of merging, pretend items of different capitalizations are the same <-------- *Note to potentially remove later*
    df_copy['item_name'] = df_copy['item_name'].str.lower()
    items_tagged_copy['item_name'] = items_tagged_copy['item_name'].str.lower()
    #items_tagged_copy.drop_duplicates(['item_name', 'location_id'], inplace=True)
    
    # Perform the merge on both 'item_name' and 'location_id'
    merged = pd.merge(df_copy.reset_index(), items_tagged_copy, 
                      on=['item_name', 'location_id'], how='left')

    # Set 'created_at' back as the index
    merged.set_index('created_at', inplace=True)

    # Remove failed merges due to missing data on 27 and encoding errors elsewhere <-------- *Note to potentially remove later*
    merged = merged[~merged['is_plant_based'].isna()]

    # Recapitalize
    merged['item_name'].str.capitalize()

    # Store in the dictionary
    merged_sales_and_menu[location_id] = merged

## Data Coverage

In [ ]:
first_start_date

In [ ]:
start_dates = []
for loc_id, df in restaurant_sales_data.items():
    start_dates.append(df.index.min())
first_start_date = pd.Series(start_dates).sort_values().iloc[0]

In [ ]:
# Find the overall date range
end_date = pd.to_datetime("now") # No timezone info because we're converting to weeks

# Remove timezone info
first_start_date = first_start_date.tz_localize(None) 

# Generate a complete range of weeks from start to end
all_weeks = pd.date_range(start=first_start_date, end=end_date, freq='W').to_period('W')

In [ ]:
active_weeks = {}

for loc_id, df in restaurant_sales_data.items():
    df = df.copy()
    df.index = df.index.tz_localize(None) 
    df['Week'] = df.index.to_period('W')
    active_weeks[loc_id] = set(df.groupby('Week').size().index.tolist())

In [ ]:
before_after_details = before_after_details.reset_index().sort_values('location_id')
before_after_details.set_index('location_id', inplace=True)

In [ ]:
before_after_details

In [ ]:
# Visualizing with gaps for inactive weeks
plt.figure(figsize=(14, 8))

for loc_id, weeks in active_weeks.items():
    for week in weeks:
        plt.hlines(y=loc_id, xmin=week.start_time, xmax=week.end_time, colors='blue', lw=2, label=loc_id)

    promo_datetime = pd.to_datetime(before_after_details.loc[loc_id]['cross_over_date'])
    plt.plot(promo_datetime, loc_id, 'ro')


plt.title('Weekly Activity for Each Restaurant with Gaps for Inactive Weeks')
plt.xlabel('Date')
plt.ylabel('Restaurant ID')
plt.yticks()
plt.tight_layout()
plt.show()

In [ ]:
two_months_prior

In [ ]:
promo_datetime + pd.DateOffset(months=2)

In [ ]:
buffer_data_list = []
for loc_id, df in restaurant_sales_data.items():
    df = df.copy()
    df.index = df.index.tz_localize(None) 
    promo_datetime = pd.to_datetime(before_after_details.loc[loc_id, 'cross_over_date'])
    before_limit = promo_datetime - pd.DateOffset(months=2)

    before = df.loc[:promo_datetime]
    two_months_prior = before.loc[before_limit:]
    before_week_count = (0 < before['item_name'].resample('W-MON').count()).sum()
    two_months_prior_week_count = (0 < two_months_prior['item_name'].resample('W-MON').count()).sum()
    before_size = before.shape[0]

    row = {'loc_id': loc_id, 'before_size': before_size, 'before_week_count': before_week_count, 'two_months_prior_week_count': two_months_prior_week_count}
    buffer_data_list.append(row)
    
buffer_data = pd.DataFrame(buffer_data_list)

In [ ]:
buffer_data

In [ ]:
# Number of customers
customers.shape[0]

In [ ]:
# Number of customers without age data
customers['age'].str.contains('nan').sum()

In [ ]:
# Number of customers without gender data
customers['gender'].str.contains('nan').sum()

In [ ]:
precomputed_customers = {}
for loc_id in location_ids:
    precomputed_customers[loc_id] = set(fdf(customers).filter('location_id', loc_id)['customer_id'].unique().tolist())

In [ ]:
for i in range(len(location_ids)):
    loc_id1 = location_ids[i]
    customer_set1 = precomputed_customers[loc_id1]
    for loc_id2 in location_ids[i:]:
        if loc_id1 != loc_id2:
            customer_set2 = precomputed_customers[loc_id2]
            intersection = customer_set1.intersection(customer_set2)
            if intersection:
                print(loc_id1, loc_id2, len(intersection))
            else:
                pass
                # print(loc_id1, loc_id2, "Nope")



In [ ]:
customers[(customers['customer_id'] == 'nan')]

In [ ]:
merged_sales_and_customers = {}
for loc_id, df in restaurant_sales_data.items():
    df = df.copy()
    df.reset_index(inplace=True)
    merged = pd.merge(df, customers, on=['location_id', 'customer_id'], how='left')

    # All entries have a batch, since it was adding after data processing
    print(merged[~merged['customer_id'].isna() & merged['batch'].isna()].size)
    print(merged[merged['batch'].isna()].size)
    merged.set_index('created_at', inplace=True, drop=False)
    merged_sales_and_customers[loc_id] = merged

In [ ]:
female_proportions_list = []

for loc_id in location_ids:

    row = {'location_id': loc_id}

    cross_over = before_after_details.loc[loc_id]['cross_over_date']

    before_genders = merged_sales_and_customers[loc_id].loc[:cross_over]['gender'].value_counts()
    after_genders = merged_sales_and_customers[loc_id].loc[cross_over:]['gender'].value_counts()
    
    if not before_genders.empty and not after_genders.empty:

        before_female_total = before_genders.loc['female']
        after_female_total = after_genders.loc['female']
        before_known_gender_total = before_genders.loc['male'] + before_genders.loc['female']
        after_known_gender_total = after_genders.loc['male'] + after_genders.loc['female']

        before_frac_female = -1
        if before_known_gender_total != 0:
            before_frac_female = before_female_total/before_known_gender_total
            
        after_frac_female = -1
        if after_known_gender_total != 0:
            after_frac_female = after_female_total/after_known_gender_total
            
        sample_qualifer = ""
        if 1000 < before_known_gender_total and 1000 < after_known_gender_total:
            sample_qualifer = "Big Enough Sample"

        # print(loc_id, round(before_frac_female*100)/100, round(after_frac_female*100)/100, before_known_gender_total, after_known_gender_total, sample_qualifer)

        row['female_proportion_before'] = round(before_frac_female*100)/100
        row['female_proportion_after'] = round(after_frac_female*100)/100
        row['enough_data'] = bool(sample_qualifer)
        row['before_known_gender_total'] = before_known_gender_total
        row['after_known_gender_total'] = after_known_gender_total

    elif before_genders.empty and after_genders.empty:

        print(loc_id, "--No customer data!")

    else:

        print(loc_id, "--Not enough data before or after.")

    female_proportions_list.append(row)



female_proportions = pd.DataFrame(female_proportions_list)


In [ ]:
female_proportions

In [ ]:
merged_sales_and_menu['EMBVNVD207CC6']

In [ ]:
fdf(items_tagged).filter('item_name', '16 monkey boy')

In [ ]:
fdf(merged_sales_and_menu['EMBVNVD207CC6']).filter('item_name', '16 monkey boy')

In [ ]:
fdf(merged_sales_and_menu['EMBVNVD207CC6']).filter('dish_category', 'alcohol', exclude=True)['item_name'].value_counts()

In [ ]:
customer_revisit_row_list = []
for loc_id, df in merged_sales_and_customers.items():
    
    row = {'location_id': loc_id, 'total': df['customer_id'].nunique()}

    for j in [1, 2, 5, 10]:

        merge_successes = df[~df['batch'].isna()]
        customers_revisits_j = merge_successes.groupby('customer_id')['created_at'].nunique() > j
        num_customer_revisits = customers_revisits_j.sum()
        row['more than ' + str(j)] = num_customer_revisits

    customer_revisit_row_list.append(row)
pd.DataFrame(customer_revisit_row_list)

## Promotional Items

In [ ]:
promo_match_list = []

for loc_id, df in merged_sales_and_menu.items():

    # Looking for promo
    promo_item = before_after_details.loc[loc_id, 'first_plant_based_mention']
    cross_over_date = before_after_details.loc[loc_id, 'cross_over_date']
    promo_df = fdf(df).filter('item_name', promo_item)
    ever_found = not promo_df.empty


    # Actual first
    plant_based = fdf(df).filter('is_plant_based', 'yes')
    first_plant_based = plant_based['item_name'].iloc[0]
    its_date = plant_based.index[0]

    # Do they match?
    is_first = promo_item.lower() == first_plant_based.lower()

    row = {'location_id': loc_id, 'promo_item': promo_item, 'cross_over_date': cross_over_date, 'ever_found': ever_found, 'is_first': is_first, 'first_plant_based': first_plant_based, 'its_date': its_date}

    promo_match_list.append(row)
    
pd.DataFrame(promo_match_list)

## Metrics

In [ ]:
location_ids[0]

In [ ]:
restaurant_sales_data[location_ids[0]].head(50)

In [ ]:
merged_sales_and_menu[location_ids[0]].resample('M')['item_name'].count().head(50)

In [ ]:
plt.plot(fdf(merged_sales_and_menu[location_ids[0]]).filter('is_plant_based','yes').resample('M')['item_name'].count() / merged_sales_and_menu[location_ids[0]].resample('M')['item_name'].count())

In [ ]:
visual_dict['0RJH3FFPYBPEY']
plt.show()

In [ ]:
weekly_pb_list = []
monthly_pb_list = []
yearly_pb_list = []

visual_dict = {}

for loc_id, df in merged_sales_and_menu.items():

    fig, ax = plt.subplots()  # Creates a single subplot


    # df['week'] = df.index.isocalendar().week
    # df['month'] = df.index.month
    # df['year'] = df.index.year
    plant_based = fdf(df).filter('is_plant_based','yes')
    promo_datetime = pd.to_datetime(before_after_details.loc[loc_id]['cross_over_date'])
    # weekly_pb_list.append(plant_based.groupby('week')['item_name'].count())
    # monthly_pb_list.append(plant_based.groupby('month')['item_name'].count())
    # yearly_pb_list.append(plant_based.groupby('year')['item_name'].count())
    print(loc_id)
    ax.plot(plant_based.resample('M')['item_name'].count() / df.resample('M')['item_name'].count())
    ax.axvline(x=promo_datetime, color='red', linestyle='--')
    ax.set_title(loc_id)
    visual_dict[loc_id] = ax


## Sales Visuals

In [ ]:
num_plots = len(merged_sales_and_menu)
cols = 4 
rows = math.ceil(num_plots / cols) * 4

# Create a figure with multiple subplots
fig, axs = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axs = axs.flatten()  # Flatten the array for easy indexing

top_dishes_number = 20

for i, (location_id, data) in tqdm(enumerate(merged_sales_and_menu.items())):

    df = data.copy()
    
    # Get the top 20 items sorted in descending order
    top_items_by_times_ordered = df['item_name'].value_counts().nlargest(top_dishes_number).sort_values()

    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i].barh(top_items_by_times_ordered.index.str.slice(0,20).str.capitalize(), top_items_by_times_ordered)
    axs[4*i].set_title(f'{location_id}\nTotal by Times Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i].tick_params(axis='y', labelsize=10)
    axs[4*i].set_xlabel('Times Ordered')

    top_items_by_quantity_ordered = df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values()
    
    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 1].barh(top_items_by_quantity_ordered.index.str.slice(0,20).str.capitalize(), top_items_by_quantity_ordered, color='cyan')
    axs[4*i + 1].set_title(f'{location_id}\nTotal by Quantity Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 1].tick_params(axis='y', labelsize=10)
    axs[4*i + 1].set_xlabel('Quantity Ordered')

    plant_df = df[df['is_plant_based'] == 'yes']
    
    # Get the top 20 items sorted in descending order
    top_plant_based_items_by_times_ordered = plant_df['item_name'].value_counts().nlargest(top_dishes_number).sort_values()

    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 2].barh(top_plant_based_items_by_times_ordered.index.str.slice(0,20).str.capitalize(), top_plant_based_items_by_times_ordered, color='green')
    axs[4*i + 2].set_title(f'{location_id}\nPlant-Based by Times Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 2].tick_params(axis='y', labelsize=10)
    axs[4*i + 2].set_xlabel('Times Ordered')

    top_plant_based_items_by_quantity_ordered = plant_df.groupby('item_name')['item_quantity'].sum().nlargest(top_dishes_number).sort_values()
    
    # Plot in the appropriate subplot as a horizontal bar chart
    axs[4*i + 3].barh(top_plant_based_items_by_quantity_ordered.index.str.slice(0,20).str.capitalize(), top_plant_based_items_by_quantity_ordered, color='lime')
    axs[4*i + 3].set_title(f'{location_id}\nPlant-Based by Quantity Ordered')
    
    # Set y-axis label and make y-labels smaller
    axs[4*i + 3].tick_params(axis='y', labelsize=10)
    axs[4*i + 3].set_xlabel('Quantity Ordered')

# Add a global title at the top of the figure
fig.suptitle('Distribution of Plant-Based and Total Item Orders in Each Restaurant', fontsize=16)

# Adjust layout and save the figure
plt.tight_layout()
plt.subplots_adjust(top=0.95)  # Adjust the top margin to make room for the global title
plt.savefig('Restaurant Both Sales Distributions.png', bbox_inches='tight')